# Repository Recommender - Data Exploration & Baseline Model

This notebook demonstrates the DS workflow for the git-query recommender:
1. Load repository dataset from the gateway API
2. Explore data distributions
3. Extract features for ranking
4. Train a baseline ranking model
5. Evaluate with IR metrics

In [3]:
import sys
sys.path.insert(0, '../../..')  # Add project root to path

import os
from pathlib import Path

from dotenv import load_dotenv
load_dotenv(dotenv_path='../../../.env', override=True)  # Load .env from project root

from src.recommender.data import RepoDataset, FeatureExtractor
import pandas as pd
import numpy as np
import fastparquet
import pyarrow
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import ndcg_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import joblib
import os

RANDOM_STATE = 42

EMBEDDING_API_KEY not set. Embedding functionality may be limited.


In [6]:
df = None

REPO_ROOT = Path.cwd().parents[3]
NOTEBOOK_DATA_DIR = Path.cwd() / "data"
REPO_DATA_DIR = REPO_ROOT / "data"

print("CWD:", Path.cwd())
print("REPO_ROOT:", REPO_ROOT)
print("NOTEBOOK_DATA_DIR exists?", NOTEBOOK_DATA_DIR.exists())
print("REPO_DATA_DIR exists?", REPO_DATA_DIR.exists())

# 1) Prefer the engineered CSV if present (this matches what you actually have)
csv_candidates = [
    NOTEBOOK_DATA_DIR / "features_engineered.csv",
    REPO_DATA_DIR / "features_engineered.csv",
]
csv_path = next((p for p in csv_candidates if p.exists()), None)

if csv_path is not None:
    print("Loading engineered CSV:", csv_path)
    df = pd.read_csv(csv_path)

# 2) Else try RepoDataset local cache (json/parquet)
if df is None:
    try:
        from src.recommender.data import RepoDataset

        local_candidates = [
            REPO_DATA_DIR / "repos.parquet",
            REPO_DATA_DIR / "repos.json",
            NOTEBOOK_DATA_DIR / "repos.parquet",
            NOTEBOOK_DATA_DIR / "repos.json",
        ]
        local_path = next((p for p in local_candidates if p.exists()), None)

        if local_path is not None:
            print("Loading via RepoDataset.from_local:", local_path)
            ds = RepoDataset.from_local(str(local_path))
            df = ds.to_dataframe()
    except Exception as e:
        print("Local RepoDataset load not used:", e)

# 3) Else try gateway if configured
if df is None:
    try:
        from src.recommender.data import RepoDataset

        GATEWAY_URL = os.getenv("GITQUERY_GATEWAY_URL") or os.getenv("GATEWAY_URL")
        API_KEY = os.getenv("GITQUERY_API_KEY") or os.getenv("API_KEY")

        if GATEWAY_URL and API_KEY:
            print("Fetching from gateway...")
            ds = RepoDataset.from_gateway(url=GATEWAY_URL, api_key=API_KEY, max_repos=5000)
            df = ds.to_dataframe()
        else:
            print("Gateway env vars not set (GITQUERY_GATEWAY_URL/GITQUERY_API_KEY).")
    except Exception as e:
        print("Gateway RepoDataset fetch not used:", e)

# Final
if df is None or len(df) == 0:
    print("Skipping EDA: df not loaded (no CSV found and no json/parquet/gateway available).")
else:
    print("Loaded df:", df.shape)
    display(df.head(5))

CWD: c:\Users\jmlar\OneDrive\Documentos\GitHub\git-query\src\recommender\notebooks
REPO_ROOT: c:\Users\jmlar\OneDrive\Documentos\GitHub
NOTEBOOK_DATA_DIR exists? True
REPO_DATA_DIR exists? False
Gateway env vars not set (GITQUERY_GATEWAY_URL/GITQUERY_API_KEY).
Skipping EDA: df not loaded (no CSV found and no json/parquet/gateway available).


In [ ]:
def _col_exists(c): 
    return c in df.columns

def eda_overview(df):
    print("Shape:", df.shape)
    display(df.head(3))

    schema = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    }).sort_values(["missing_%", "n_unique"], ascending=False)

    display(schema.head(40))
    return schema

In [ ]:
def eda_missingness(df, top_k=25):
    miss = df.isna().mean().sort_values(ascending=False).head(top_k)
    plt.figure(figsize=(10,4))
    miss.plot(kind="bar")
    plt.title(f"Missingness rate (top {top_k})")
    plt.ylabel("fraction missing")
    plt.tight_layout()
    plt.show()

In [ ]:
def eda_duplicates(df):
    keys = [c for c in ["id", "full_name", "name", "html_url", "clone_url"] if c in df.columns]
    if not keys:
        print("No standard id/url/name keys found for duplicate check.")
        return
    for k in keys:
        print(f"Duplicates by {k}: {int(df[k].duplicated().sum())}")

In [ ]:
def eda_numeric(df, max_cols=20):
    num = df.select_dtypes(include=[np.number])
    if num.shape[1] == 0:
        print("No numeric columns.")
        return
    display(num.describe(percentiles=[.5,.75,.9,.95,.99]).T.head(max_cols))

    # quick hist for a few likely columns if present
    likely = [c for c in ["stars","forks","watchers","open_issues","contributors","topics_count",
                          "readme_word_count","days_since_push"] if c in num.columns]
    for c in likely[:6]:
        plt.figure(figsize=(8,3))
        num[c].replace([np.inf, -np.inf], np.nan).dropna().clip(lower=0).hist(bins=50)
        plt.title(f"{c} distribution")
        plt.tight_layout()
        plt.show()

In [ ]:
def eda_categorical(df, cols=None, top_k=20):
    if cols is None:
        cols = [c for c in ["language","license"] if c in df.columns]
    for c in cols:
        vc = df[c].fillna("missing").astype(str).value_counts().head(top_k)
        plt.figure(figsize=(10,4))
        vc.plot(kind="bar")
        plt.title(f"Top {top_k}: {c}")
        plt.tight_layout()
        plt.show()
        display(vc.to_frame("count"))

In [ ]:
def eda_correlation(df, max_cols=12):
    num = df.select_dtypes(include=[np.number]).copy()
    if num.shape[1] < 2:
        print("Not enough numeric columns for correlation.")
        return
    # keep only columns with variance
    keep = [c for c in num.columns if num[c].nunique(dropna=True) > 1]
    num = num[keep]
    if num.shape[1] < 2:
        print("Not enough non-constant numeric columns for correlation.")
        return
    # limit
    cols = keep[:max_cols]
    corr = num[cols].corr()
    plt.figure(figsize=(8,6))
    plt.imshow(corr, aspect="auto")
    plt.xticks(range(len(cols)), cols, rotation=90)
    plt.yticks(range(len(cols)), cols)
    plt.colorbar()
    plt.title("Correlation (subset)")
    plt.tight_layout()
    plt.show()

In [ ]:
def eda_text(df):
    # Works for either raw or engineered
    for c in ["readme", "description", "text"]:
        if c in df.columns:
            s = df[c].fillna("").astype(str)
            lengths = s.str.len()
            plt.figure(figsize=(8,3))
            lengths.clip(upper=lengths.quantile(0.99)).hist(bins=50)
            plt.title(f"{c} length (clipped p99)")
            plt.tight_layout()
            plt.show()
            print(f"{c}: empty rate =", float((s.str.strip() == "").mean()))

In [ ]:
def eda_topic_like(df):
    # handles topics_list, topics_str, topics (list-as-string), etc.
    if "topics_list" in df.columns:
        lens = df["topics_list"].apply(lambda x: len(x) if isinstance(x, list) else 0)
        plt.figure(figsize=(8,3))
        lens.hist(bins=30)
        plt.title("topics_list length")
        plt.tight_layout()
        plt.show()
    elif "topics_str" in df.columns:
        lens = df["topics_str"].fillna("").astype(str).apply(lambda x: 0 if x.strip()=="" else len(x.split(",")))
        plt.figure(figsize=(8,3))
        lens.hist(bins=30)
        plt.title("topics_str count (comma-split)")
        plt.tight_layout()
        plt.show()
    elif "topics" in df.columns:
        # try to parse if it's a string representation
        s = df["topics"].fillna("")
        lens = s.apply(lambda x: len(x) if isinstance(x, list) else (len(str(x).split(",")) if str(x).strip() else 0))
        plt.figure(figsize=(8,3))
        lens.hist(bins=30)
        plt.title("topics count (best-effort)")
        plt.tight_layout()
        plt.show()

In [ ]:
def eda_leakage_checks(df):
    # helpful because your CSV is "engineered"
    leak = [c for c in df.columns if any(k in c.lower() for k in ["query", "overlap", "relevance", "label", "target"])]
    print("Leakage/label-ish columns:", leak[:50])

    const = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    print("Constant columns:", const[:50])

    nan_top = df.isna().mean().sort_values(ascending=False).head(20)
    display(nan_top.to_frame("missing_fraction"))

In [ ]:
if df is None or len(df) == 0:
    print("No df loaded, cannot run EDA.")
else:
    schema = eda_overview(df)
    eda_missingness(df)
    eda_duplicates(df)
    eda_leakage_checks(df)

    eda_numeric(df)
    eda_categorical(df)
    eda_correlation(df)

    eda_text(df)
    eda_topic_like(df)

In [1]:
# Store cleaned EDA dataset into MongoDB collection: repositories
import os
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(dotenv_path='../../../.env', override=True)
load_dotenv(dotenv_path='.env', override=False)

if 'df' not in globals() or df is None or len(df) == 0:
    print('No DataFrame available. Run data loading + EDA cells first.')
else:
    work_df = df.copy()

    # Normalize common null representations and keep values JSON-safe.
    work_df = work_df.replace({np.nan: None})
    if 'full_name' in work_df.columns:
        work_df['full_name'] = work_df['full_name'].astype(str).str.strip()

    def _is_missing(v):
        if v is None:
            return True
        if isinstance(v, float) and pd.isna(v):
            return True
        if isinstance(v, str) and v.strip() == '':
            return True
        return False

    def _clean_value(v):
        if isinstance(v, (np.integer,)):
            return int(v)
        if isinstance(v, (np.floating,)):
            if pd.isna(v):
                return None
            return float(v)
        if isinstance(v, (np.bool_,)):
            return bool(v)
        if isinstance(v, pd.Timestamp):
            return v.to_pydatetime().isoformat()
        if isinstance(v, datetime):
            return v.isoformat()
        if isinstance(v, np.ndarray):
            return v.tolist()
        if isinstance(v, (set, tuple)):
            return list(v)
        if isinstance(v, dict):
            return {str(k): _clean_value(val) for k, val in v.items()}
        if isinstance(v, list):
            return [_clean_value(x) for x in v]
        return v

    def _build_doc(row):
        doc = {k: _clean_value(v) for k, v in row.items()}

        # Stable identity for upserts into repositories.
        full_name = doc.get('full_name')
        repo_id = doc.get('id')
        name = doc.get('name')
        if not _is_missing(full_name):
            _id = str(full_name).strip()
        elif not _is_missing(repo_id):
            _id = str(repo_id).strip()
        elif not _is_missing(name):
            _id = str(name).strip()
        else:
            return None

        doc['_id'] = _id
        doc['cleaned_at'] = datetime.now(timezone.utc).isoformat()
        doc['data_source'] = 'notebook_eda_cleaned'
        return doc

    docs = []
    for row in work_df.to_dict(orient='records'):
        doc = _build_doc(row)
        if doc is not None:
            docs.append(doc)

    # Deduplicate by _id (last occurrence wins).
    deduped = {doc['_id']: doc for doc in docs}
    docs_to_write = list(deduped.values())

    gateway_url = (
        os.getenv('GITQUERY_GATEWAY_URL')
        or os.getenv('GATEWAY_URL')
        or os.getenv('API_BASE_URL')
    )
    api_key = (
        os.getenv('GITQUERY_API_KEY')
        or os.getenv('APIKEY_MONGODB')
        or os.getenv('API_KEY')
        or os.getenv('GATEWAY_API_KEY')
    )

    if not gateway_url or not api_key:
        print('Gateway credentials missing.')
        print('Need one URL var: GITQUERY_GATEWAY_URL | GATEWAY_URL | API_BASE_URL')
        print('Need one KEY var: GITQUERY_API_KEY | APIKEY_MONGODB | API_KEY | GATEWAY_API_KEY')
    elif len(docs_to_write) == 0:
        print('No valid documents to write after cleaning/deduplication.')
    else:
        bulk_url = f"{gateway_url.rstrip('/')}/api/mongodb/collections/repositories/bulk"
        query_url = f"{gateway_url.rstrip('/')}/api/mongodb/collections/repositories/query"
        headers = {'X-API-Key': api_key, 'Content-Type': 'application/json'}

        batch_size = 1000
        inserted_total = 0
        updated_total = 0
        errors_total = 0

        for i in range(0, len(docs_to_write), batch_size):
            batch = docs_to_write[i : i + batch_size]
            payload = {'documents': batch, 'ordered': False, 'upsert': True}
            try:
                resp = requests.post(bulk_url, headers=headers, json=payload, timeout=60)
                resp.raise_for_status()
                data = resp.json()
                inserted_total += int(data.get('inserted', 0) or 0)
                updated_total += int(data.get('updated', 0) or 0)
                errors_total += len(data.get('errors', []) or [])
            except Exception as exc:
                errors_total += len(batch)
                print(f'Batch failed at rows {i}..{i + len(batch) - 1}: {exc}')

        # Verify by reading count/sample from repositories.
        verify_payload = {
            'filter': {},
            'projection': {'_id': 1, 'full_name': 1, 'name': 1, 'language': 1, 'cleaned_at': 1},
            'limit': 3,
            'skip': 0
        }
        verify_resp = requests.post(query_url, headers=headers, json=verify_payload, timeout=30)
        verify_count = None
        verify_docs = []
        if verify_resp.ok:
            verify_json = verify_resp.json()
            verify_count = verify_json.get('count')
            verify_docs = verify_json.get('documents', [])

        print('Prepared docs:', len(docs))
        print('Deduped docs:', len(docs_to_write))
        print('Upsert inserted:', inserted_total)
        print('Upsert updated:', updated_total)
        print('Upsert batch errors:', errors_total)
        print('repositories sample count returned:', verify_count)
        print('repositories sample docs:')
        print(json.dumps(verify_docs, indent=2, ensure_ascii=True))

No DataFrame available. Run data loading + EDA cells first.


## Next Steps

- [ ] Try LightGBM ranker with `lambdarank` objective
- [ ] Add more features (readme analysis, contributor count, issue stats)
- [ ] Test with more queries and build a proper evaluation set
- [ ] Compare pretrained cross-encoder vs custom model
- [ ] Export trained model for production serving

In [ ]:
# LightGBM LambdaRank training cell
# Trains a LightGBM reranker using FeatureExtractor and saves artifact + registry JSON.
import os, time, json
import joblib
import lightgbm as lgb
import numpy as np
from datetime import datetime, timezone
from src.recommender.data.features import FeatureExtractor
from src.recommender.config import settings

# Ensure we have data loaded by earlier cells (df)
if 'df' not in globals() or df is None or len(df) == 0:
    print('No repository DataFrame (df) available for training. Fill df via the notebook data loading cells first.')
else:
    import pandas as pd

    fe = FeatureExtractor()
    rs = int(globals().get('RANDOM_STATE', 42))

    work_df = df.copy()

    # Add fallbacks so FeatureExtractor can run on engineered CSVs
    if 'name' not in work_df.columns:
        work_df['name'] = ''
    if 'description' not in work_df.columns:
        work_df['description'] = ''
    if 'license' not in work_df.columns:
        work_df['license'] = ''
    if 'topics' not in work_df.columns:
        work_df['topics'] = [[] for _ in range(len(work_df))]

    # Option 3: Use language as a proxy query when topics are unavailable
    if 'language' not in work_df.columns:
        print("'language' column not found; cannot build proxy query groups for training.")
    else:
        work_df['query'] = work_df['language'].fillna('missing').astype(str).str.strip().str.lower()
        work_df = work_df[work_df['query'] != ''].reset_index(drop=True)

        if len(work_df) < 50:
            print('Not enough examples to build per-query groups.')
        else:
            top_queries = work_df['query'].value_counts().head(100).index.tolist()
            rows = []
            qid = 0

            for q in top_queries:
                q_df = work_df[work_df['query'] == q]
                if len(q_df) < 5:
                    continue

                candidates = q_df.sample(n=min(100, len(q_df)), replace=False, random_state=rs)
                neg_pool = work_df[work_df['query'] != q]
                if len(neg_pool) == 0:
                    continue

                neg_n = min(100, max(20, len(candidates) // 2), len(neg_pool))
                negs = neg_pool.sample(n=neg_n, replace=False, random_state=rs)

                pool = pd.concat([candidates, negs], ignore_index=True).reset_index(drop=True)
                pool['query_id'] = qid
                pool['query_text'] = q
                rows.append(pool)
                qid += 1

            if not rows:
                print('Could not build any query groups for training.')
            else:
                train_df = pd.concat(rows, ignore_index=True)

                def build_rank_data(frame):
                    feature_frames = []
                    labels = []
                    group_sizes = []

                    for _, grp in frame.groupby('query_id'):
                        qtext = grp['query_text'].iloc[0]
                        feats = fe.extract_all(grp, query=qtext)

                        # Proxy relevance from stars within each group (top quartile = relevant)
                        stars = pd.to_numeric(grp.get('stars', 0), errors='coerce').fillna(0)
                        threshold = stars.quantile(0.75)
                        rel = (stars >= threshold).astype(int)
                        if rel.sum() == 0:
                            rel = (stars == stars.max()).astype(int)

                        feature_frames.append(feats)
                        labels.append(rel)
                        group_sizes.append(len(feats))

                    X_part = pd.concat(feature_frames, axis=0).reset_index(drop=True)
                    y_part = pd.concat(labels, axis=0).reset_index(drop=True)
                    return X_part, y_part, group_sizes

                # Query-level holdout split to avoid leakage across same query_id
                unique_qids = np.array(sorted(train_df['query_id'].unique()))
                rng = np.random.default_rng(rs)
                rng.shuffle(unique_qids)

                has_holdout = len(unique_qids) >= 5
                if has_holdout:
                    n_val_q = max(1, int(len(unique_qids) * 0.2))
                    val_qids = set(unique_qids[:n_val_q].tolist())
                    train_qids = set(unique_qids[n_val_q:].tolist())

                    train_part = train_df[train_df['query_id'].isin(train_qids)].reset_index(drop=True)
                    val_part = train_df[train_df['query_id'].isin(val_qids)].reset_index(drop=True)

                    X_train, y_train, groups_train = build_rank_data(train_part)
                    X_val, y_val, groups_val = build_rank_data(val_part)
                else:
                    X_train, y_train, groups_train = build_rank_data(train_df)
                    X_val, y_val, groups_val = None, None, None

                feature_cols = X_train.columns.tolist()
                X_train = X_train.reindex(columns=feature_cols, fill_value=0)

                if X_val is not None:
                    X_val = X_val.reindex(columns=feature_cols, fill_value=0)

                X = X_train
                y = y_train
                groups = groups_train

                print('Train rows:', X_train.shape, 'Feature cols:', len(feature_cols), 'Train groups:', len(groups_train))
                if X_val is not None:
                    print('Val rows:', X_val.shape, 'Val groups:', len(groups_val))

                params = {
                    'objective': 'lambdarank',
                    'metric': 'ndcg',
                    'ndcg_eval_at': [1, 5, 10],
                    'learning_rate': 0.05,
                    'num_leaves': 31,
                    'verbose': -1,
                    'seed': rs,
                }

                lgb_train = lgb.Dataset(X_train.values, label=y_train.values, group=groups_train)

                if X_val is not None:
                    lgb_valid = lgb.Dataset(X_val.values, label=y_val.values, group=groups_val, reference=lgb_train)
                    print('Starting LightGBM training with holdout early stopping...')
                    bst = lgb.train(
                        params,
                        lgb_train,
                        num_boost_round=500,
                        valid_sets=[lgb_valid],
                        valid_names=['valid'],
                        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
                    )
                    X_eval, y_eval, eval_groups = X_val, y_val, groups_val
                    eval_split = 'valid'
                else:
                    print('Starting LightGBM training (no holdout; too few query groups)...')
                    bst = lgb.train(params, lgb_train, num_boost_round=200)
                    X_eval, y_eval, eval_groups = X_train, y_train, groups_train
                    eval_split = 'train'

                # attach feature columns to model for inference adapter
                try:
                    setattr(bst, 'feature_cols', feature_cols)
                except Exception:
                    pass

                ts = int(time.time())
                model_name = f'lightgbm_lambdarank_{ts}.pkl'
                os.makedirs(settings.model_path, exist_ok=True)
                save_path = os.path.join(settings.model_path, model_name)
                joblib.dump(bst, save_path)
                print('Saved LightGBM model to', save_path)

                # Write a simple file-based registry entry (training-side)
                registry_dir = os.path.join(settings.model_path, 'metadata')
                os.makedirs(registry_dir, exist_ok=True)
                reg = {
                    'model_id': f'lightgbm_{ts}',
                    'model_type': 'reranker',
                    'variant': 'lightgbm',
                    'path': model_name,
                    'created_at': datetime.now(timezone.utc).isoformat(),
                    'feature_cols': feature_cols,
                    'eval_split': eval_split,
                }
                with open(os.path.join(registry_dir, 'model_registry_latest.json'), 'w') as fh:
                    json.dump(reg, fh, indent=2)

                print('Wrote training-side registry JSON to', os.path.join(registry_dir, 'model_registry_latest.json'))

In [ ]:
# Evaluation snippet (per-query NDCG@10 on holdout when available)
from sklearn.metrics import ndcg_score
import numpy as np

if 'bst' in globals() and 'X_eval' in globals() and 'y_eval' in globals() and 'eval_groups' in globals():
    start = 0
    ndcgs = []
    for g in eval_groups:
        end = start + g
        y_true = y_eval.values[start:end].reshape(1, -1)
        y_pred = bst.predict(X_eval.values[start:end]).reshape(1, -1)
        try:
            nd = ndcg_score(y_true, y_pred, k=min(10, g))
        except Exception:
            nd = float('nan')
        ndcgs.append(nd)
        start = end

    print('Eval split:', globals().get('eval_split', 'unknown'))
    print('Groups evaluated:', len(eval_groups))
    print('Mean NDCG@10:', float(np.nanmean(ndcgs)))
    print('Median NDCG@10:', float(np.nanmedian(ndcgs)))
else:
    print('No trained/eval model context found. Run the training cell first.')

In [ ]:
# Repeated holdout evaluation across random seeds (stability check)
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import ndcg_score
from src.recommender.data.features import FeatureExtractor

if 'train_df' not in globals() or train_df is None or len(train_df) == 0:
    print('No train_df found. Run the training cell first.')
else:
    fe_multi = FeatureExtractor()

    def _build_rank_data(frame):
        feature_frames = []
        labels = []
        group_sizes = []

        for _, grp in frame.groupby('query_id'):
            qtext = grp['query_text'].iloc[0]
            feats = fe_multi.extract_all(grp, query=qtext)

            stars = pd.to_numeric(grp.get('stars', 0), errors='coerce').fillna(0)
            threshold = stars.quantile(0.75)
            rel = (stars >= threshold).astype(int)
            if rel.sum() == 0:
                rel = (stars == stars.max()).astype(int)

            feature_frames.append(feats)
            labels.append(rel)
            group_sizes.append(len(feats))

        X_part = pd.concat(feature_frames, axis=0).reset_index(drop=True)
        y_part = pd.concat(labels, axis=0).reset_index(drop=True)
        return X_part, y_part, group_sizes

    def _ndcg_by_group(model, X_eval_local, y_eval_local, eval_groups_local):
        start = 0
        scores = []
        for g in eval_groups_local:
            end = start + g
            y_true = y_eval_local.values[start:end].reshape(1, -1)
            y_pred = model.predict(X_eval_local.values[start:end]).reshape(1, -1)
            try:
                nd = ndcg_score(y_true, y_pred, k=min(10, g))
            except Exception:
                nd = float('nan')
            scores.append(nd)
            start = end
        return float(np.nanmean(scores)), float(np.nanmedian(scores))

    seed_list = [7, 21, 42, 84, 168]
    results = []

    unique_qids_all = np.array(sorted(train_df['query_id'].unique()))
    if len(unique_qids_all) < 5:
        print('Not enough query groups for repeated holdout (need at least 5).')
    else:
        for seed in seed_list:
            rng = np.random.default_rng(seed)
            qids = unique_qids_all.copy()
            rng.shuffle(qids)

            n_val_q = max(1, int(len(qids) * 0.2))
            val_qids = set(qids[:n_val_q].tolist())
            train_qids = set(qids[n_val_q:].tolist())

            train_part = train_df[train_df['query_id'].isin(train_qids)].reset_index(drop=True)
            val_part = train_df[train_df['query_id'].isin(val_qids)].reset_index(drop=True)

            X_train_s, y_train_s, groups_train_s = _build_rank_data(train_part)
            X_val_s, y_val_s, groups_val_s = _build_rank_data(val_part)

            feature_cols_s = X_train_s.columns.tolist()
            X_train_s = X_train_s.reindex(columns=feature_cols_s, fill_value=0)
            X_val_s = X_val_s.reindex(columns=feature_cols_s, fill_value=0)

            ds_train = lgb.Dataset(X_train_s.values, label=y_train_s.values, group=groups_train_s)
            ds_val = lgb.Dataset(X_val_s.values, label=y_val_s.values, group=groups_val_s, reference=ds_train)

            params = {
                'objective': 'lambdarank',
                'metric': 'ndcg',
                'ndcg_eval_at': [1, 5, 10],
                'learning_rate': 0.05,
                'num_leaves': 31,
                'verbose': -1,
                'seed': seed,
            }

            model = lgb.train(
                params,
                ds_train,
                num_boost_round=500,
                valid_sets=[ds_val],
                valid_names=['valid'],
                callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
            )

            mean_ndcg, median_ndcg = _ndcg_by_group(model, X_val_s, y_val_s, groups_val_s)
            results.append({
                'seed': seed,
                'val_groups': len(groups_val_s),
                'mean_ndcg_at_10': mean_ndcg,
                'median_ndcg_at_10': median_ndcg,
                'best_iteration': int(model.best_iteration) if model.best_iteration else None,
            })

        results_df = pd.DataFrame(results)
        display(results_df)
        print('Aggregate mean NDCG@10:', float(results_df['mean_ndcg_at_10'].mean()))
        print('Aggregate std NDCG@10:', float(results_df['mean_ndcg_at_10'].std(ddof=0)))
        print('Aggregate median NDCG@10:', float(results_df['median_ndcg_at_10'].median()))

In [7]:
# Force Mongo-backed fetch via gateway for this run
import os
from src.recommender.data import RepoDataset

GATEWAY_URL = (
    os.getenv("GITQUERY_GATEWAY_URL")
    or os.getenv("GATEWAY_URL")
    or os.getenv("API_BASE_URL")
)
API_KEY = (
    os.getenv("GITQUERY_API_KEY")
    or os.getenv("APIKEY_MONGODB")
    or os.getenv("API_KEY")
    or os.getenv("GATEWAY_API_KEY")
)

if not GATEWAY_URL or not API_KEY:
    print("Gateway credentials missing.")
    print("Checked URL vars: GITQUERY_GATEWAY_URL, GATEWAY_URL, API_BASE_URL")
    print("Checked KEY vars: GITQUERY_API_KEY, APIKEY_MONGODB, API_KEY, GATEWAY_API_KEY")
else:
    print("Fetching from gateway (Mongo-backed):", GATEWAY_URL)
    ds = RepoDataset.from_gateway(url=GATEWAY_URL, api_key=API_KEY, max_repos=5_000_000, batch_size=1000)
    df = ds.to_dataframe()
    print("Loaded from gateway df:", df.shape)
    display(df.head(3))

Fetching from gateway (Mongo-backed): https://gitquery.davidhoerz.com
[RepoDataset] Count API returned 1 -- using fallback (will stop on empty batch).
[RepoDataset] Target repo count: 5000000
[RepoDataset] Fetched batch 1 (1000 docs, total so far: 1000)
[RepoDataset] Fetched batch 2 (1000 docs, total so far: 2000)
[RepoDataset] Fetched batch 3 (1000 docs, total so far: 3000)
[RepoDataset] Fetched batch 4 (1000 docs, total so far: 4000)
[RepoDataset] Fetched batch 5 (1000 docs, total so far: 5000)
[RepoDataset] Fetched batch 6 (1000 docs, total so far: 6000)
[RepoDataset] Fetched batch 7 (1000 docs, total so far: 7000)
[RepoDataset] Fetched batch 8 (1000 docs, total so far: 8000)
[RepoDataset] Fetched batch 9 (1000 docs, total so far: 9000)
[RepoDataset] Fetched batch 10 (1000 docs, total so far: 10000)
[RepoDataset] Fetched batch 11 (1000 docs, total so far: 11000)
[RepoDataset] Fetched batch 12 (1000 docs, total so far: 12000)
[RepoDataset] Fetched batch 13 (1000 docs, total so far: 1

KeyboardInterrupt: 

In [ ]:
# Env diagnostics (safe: does not print secret values)
import os

url_vars = ["GITQUERY_GATEWAY_URL", "GATEWAY_URL", "API_BASE_URL"]
key_vars = ["GITQUERY_API_KEY", "APIKEY_MONGODB", "API_KEY", "GATEWAY_API_KEY"]

present = {}
for var in url_vars + key_vars:
    value = os.getenv(var)
    present[var] = value is not None and str(value).strip() != ""

print("Gateway URL vars:")
for var in url_vars:
    print(f"  - {var}: {'SET' if present[var] else 'MISSING'}")

print("Gateway key vars:")
for var in key_vars:
    print(f"  - {var}: {'SET' if present[var] else 'MISSING'}")

resolved_url_var = next((v for v in url_vars if present[v]), None)
resolved_key_var = next((v for v in key_vars if present[v]), None)

print("Resolved URL source:", resolved_url_var if resolved_url_var else "None")
print("Resolved KEY source:", resolved_key_var if resolved_key_var else "None")

if not resolved_url_var or not resolved_key_var:
    print("\nMissing required env vars for Mongo-backed gateway fetch.")
    print("Set one URL var + one KEY var, then rerun the fetch/training cells.")

In [ ]:
# Run summary metrics dashboard
import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score

if not all(name in globals() for name in ["bst", "X_train", "y_train", "groups_train", "X_eval", "y_eval", "eval_groups"]):
    print("Missing training/eval context. Run fetch + training + eval cells first.")
else:
    def _group_ndcg_list(model, X_part, y_part, group_sizes, k=10):
        values = []
        start = 0
        for g in group_sizes:
            end = start + g
            y_true = y_part.values[start:end].reshape(1, -1)
            y_pred = model.predict(X_part.values[start:end]).reshape(1, -1)
            try:
                values.append(float(ndcg_score(y_true, y_pred, k=min(k, g))))
            except Exception:
                values.append(float("nan"))
            start = end
        return values

    train_ndcgs = _group_ndcg_list(bst, X_train, y_train, groups_train, k=10)
    eval_ndcgs = _group_ndcg_list(bst, X_eval, y_eval, eval_groups, k=10)

    summary = pd.DataFrame([
        {"metric": "df_rows", "value": int(len(df)) if "df" in globals() and df is not None else np.nan},
        {"metric": "df_cols", "value": int(df.shape[1]) if "df" in globals() and df is not None else np.nan},
        {"metric": "train_rows", "value": int(X_train.shape[0])},
        {"metric": "train_features", "value": int(X_train.shape[1])},
        {"metric": "train_groups", "value": int(len(groups_train))},
        {"metric": "val_rows", "value": int(X_eval.shape[0])},
        {"metric": "val_groups", "value": int(len(eval_groups))},
        {"metric": "train_positive_rate", "value": float(y_train.mean())},
        {"metric": "val_positive_rate", "value": float(y_eval.mean())},
        {"metric": "train_ndcg10_mean", "value": float(np.nanmean(train_ndcgs))},
        {"metric": "train_ndcg10_median", "value": float(np.nanmedian(train_ndcgs))},
        {"metric": "val_ndcg10_mean", "value": float(np.nanmean(eval_ndcgs))},
        {"metric": "val_ndcg10_median", "value": float(np.nanmedian(eval_ndcgs))},
        {"metric": "val_ndcg10_std", "value": float(np.nanstd(eval_ndcgs))},
        {"metric": "best_iteration", "value": int(getattr(bst, "best_iteration", 0) or 0)},
    ])

    display(summary)

    group_table = pd.DataFrame({
        "split": ["train"] * len(train_ndcgs) + [str(globals().get("eval_split", "valid"))] * len(eval_ndcgs),
        "group_ndcg10": train_ndcgs + eval_ndcgs,
    })
    display(group_table.groupby("split")["group_ndcg10"].describe())